In [4]:
import logging
import time
import pandas as pd
import redis  # Cliente síncrono estándar

# Configuración de Logs
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Parámetros de Valkey
VALKEY_HOST = "192.168.0.20"  # Cambia esto si tu Valkey no está en localhost
VALKEY_PORT = 6379
TTL_SEGUNDOS = 2592000  # 30 días de expiración
BATCH_SIZE = 5000       # Procesamos en lotes de 5000 para no saturar la red


def ingestar_hashes_usuario(ruta_archivo: str, es_parquet: bool = True):
    start_time = time.time()

    # 1. CARGA DEL DATASET
    logger.info(f"Cargando archivo desde {ruta_archivo}...")
    try:
        df = pd.read_parquet(ruta_archivo) if es_parquet else pd.read_csv(ruta_archivo)
    except Exception as e:
        logger.error(f"Error al leer el archivo: {e}")
        return

    # Verificación de columnas obligatorias
    columnas_requeridas = ["id_usuario", "email_verificado", "dias_antiguedad_cuenta", "pais_emision", "paso_3d_secure"]
    for col in columnas_requeridas:
        if col not in df.columns:
            logger.error(f"Falta la columna requerida en el dataset: '{col}'")
            return

    # 2. CONEXIÓN A VALKEY
    try:
        client = redis.Redis(host=VALKEY_HOST, port=VALKEY_PORT, decode_responses=True, socket_timeout=5)
        client.ping()
    except redis.ConnectionError:
        logger.error("No se pudo conectar a Valkey. Revisa si el servicio está corriendo.")
        return

    # 3. INGESTA MASIVA USANDO PIPELINE (HSET)
    logger.info(f"Iniciando ingesta de perfiles en lotes de {BATCH_SIZE}...")
    try:
        pipeline = client.pipeline(transaction=False)
        contador = 0

        for _, row in df.iterrows():
            id_usuario = row["id_usuario"]
            key = f"usuario:{id_usuario}"

            # Construimos el diccionario con la estructura exacta que me pediste
            # Nota: Convertimos a str/int nativo de Python porque a veces los tipos de Pandas fallan al serializar
            perfil_usuario = {
                "email_verificado": int(row["email_verificado"]),       # 1 o 0
                "dias_antiguedad_cuenta": int(row["dias_antiguedad_cuenta"]), # Valor entero
                "pais_emision": str(row["pais_emision"]),               # Ej: "ES"
                "paso_3d_secure": int(row["paso_3d_secure"]),           # 1 o 0
                # "intentos": 0,  <- Estos dos comentados si ya los manejas 
                # "bloqueado": 0,    en la lógica que tenías implementada
            }

            # Usamos HSET para guardar múltiples campos dentro del Hash de la clave
            pipeline.hset(key, mapping=perfil_usuario)
            # Mantenemos el TTL para que expire la clave completa si pasa el tiempo
            pipeline.expire(key, TTL_SEGUNDOS)
            
            contador += 1

            # Ejecución por lotes
            if contador % BATCH_SIZE == 0:
                pipeline.execute()
                logger.info(f"Progreso: {contador} usuarios indexados.")

        # Enviamos el resto
        if contador % BATCH_SIZE != 0:
            pipeline.execute()

        duracion = time.time() - start_time
        logger.info(f"¡Ingesta exitosa! {contador} perfiles estructurados en {duracion:.2f} segundos.")

    except Exception as e:
        logger.error(f"Error durante el proceso: {e}")


if __name__ == "__main__":
    # Define aquí tu archivo de variables de usuario
    RUTA_DATASET = "../data/processed/base_users.csv"  # Cambia esto a tu ruta real
    
    # Ejecutamos la ingesta
    ingestar_hashes_usuario(ruta_archivo=RUTA_DATASET, es_parquet=False)

2026-05-21 18:05:40,227 - INFO - Cargando archivo desde ../data/processed/base_users.csv...
2026-05-21 18:05:40,257 - INFO - Iniciando ingesta de perfiles en lotes de 5000...
2026-05-21 18:05:40,543 - INFO - Progreso: 5000 usuarios indexados.
2026-05-21 18:05:40,848 - INFO - ¡Ingesta exitosa! 9999 perfiles estructurados en 0.62 segundos.
